# AI App Reviewer — Autonomous Multi-Agent UX & Accessibility Auditor (with Gradio UI)

A LangGraph-orchestrated multi-agent system that takes a public URL, crawls it with Playwright,
evaluates UX and WCAG 2.2 accessibility, and produces a Markdown/JSON/PDF audit report with a
human-in-the-loop approval step — now exposed through a Gradio web UI.

**Scope note:** this notebook is scoped to run end-to-end in Google Colab (no Docker, FastAPI,
Streamlit, or test suite). `GPT-5.2` referenced in some specs isn't a real OpenAI model name at time
of writing, so `MODEL_NAME` below defaults to `gpt-4o` — swap it for whichever model your API key has
access to.

Run cells top to bottom, then use the Gradio app at the bottom: click **Run Audit**, review the
summary, then click **Approve / Reject / Re-crawl**.

In [1]:
!pip -q install langgraph langgraph-checkpoint-sqlite langchain langchain-openai langchain-community \
    beautifulsoup4 reportlab nest_asyncio playwright pydantic gradio
!playwright install --with-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

In [2]:
import os, re, json, time, uuid, asyncio, logging, sqlite3
from datetime import datetime
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional

# NOTE: we intentionally do NOT call nest_asyncio.apply() here.
# It patches asyncio.run() globally, which conflicts with the uvicorn
# server Gradio starts in demo.launch() on Python 3.12 (TypeError:
# ...got an unexpected keyword argument 'loop_factory'). Playwright's
# event loop is instead isolated in its own thread below (see run_async_in_thread).
import threading

from getpass import getpass
# Using OpenRouter (free-tier models) instead of OpenAI directly.
# Get a key at https://openrouter.ai/keys -- it looks like "sk-or-v1-...".
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OpenRouter API key (sk-or-v1-...): ")

import gradio as gr
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ai_app_reviewer")

BASE_DIR = Path("ai_app_reviewer_output")
SCREENSHOT_DIR = BASE_DIR / "screenshots"
REPORT_DIR = BASE_DIR / "reports"
LOG_DIR = BASE_DIR / "logs"
for d in [BASE_DIR, SCREENSHOT_DIR, REPORT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Specific ":free" model slugs get delisted/repriced often, so this uses
# OpenRouter's Free Models Router instead -- it auto-picks whichever free
# model is currently available, so the notebook keeps working even after
# individual free models disappear. See https://openrouter.ai/openrouter/free
# If you specifically want one named free model instead, check
# https://openrouter.ai/models?max_price=0 for a live ":free" slug.
MODEL_NAME = "openrouter/free"
llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    default_headers={
        # OpenRouter asks for these for free-tier routing/attribution; harmless if ignored.
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "AI App Reviewer",
    },
)

Enter OpenRouter API key (sk-or-v1-...): ··········


## Guardrails
URL validation, prompt-injection detection on scraped page text, PII masking, and a retry decorator for flaky browser actions.

In [3]:
MAX_PAGES_TO_CRAWL = 5
PAGE_TIMEOUT_MS = 15000
MAX_RETRIES = 2

INJECTION_PATTERNS = [
    r"ignore (all|previous|prior) instructions",
    r"system prompt",
    r"you are now",
    r"disregard (all|previous) rules",
    r"reveal your (instructions|prompt)",
]

def validate_url(url: str) -> str:
    pattern = re.compile(r"^https?://[^\s/$.?#].[^\s]*$", re.IGNORECASE)
    if not pattern.match(url):
        raise ValueError(f"Invalid URL format: {url}")
    if any(host in url for host in ["localhost", "127.0.0.1", "0.0.0.0", "169.254."]):
        raise ValueError("URLs pointing to local/internal network addresses are not allowed.")
    return url

def detect_prompt_injection(text: str) -> bool:
    lowered = text.lower()
    return any(re.search(p, lowered) for p in INJECTION_PATTERNS)

def mask_pii(text: str) -> str:
    text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", "[EMAIL_REDACTED]", text)
    text = re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "[PHONE_REDACTED]", text)
    text = re.sub(r"\b(?:\d[ -]*?){13,16}\b", "[CARD_REDACTED]", text)
    return text

def with_retry(fn):
    async def wrapper(*args, **kwargs):
        last_exc = None
        for attempt in range(1, MAX_RETRIES + 2):
            try:
                return await fn(*args, **kwargs)
            except Exception as e:
                last_exc = e
                logger.warning(f"Attempt {attempt} failed for {fn.__name__}: {e}")
                await asyncio.sleep(1.5 * attempt)
        raise last_exc
    return wrapper

## Shared graph state

In [4]:
class AppState(TypedDict, total=False):
    url: str
    plan: Dict[str, Any]
    visited_pages: List[Dict[str, Any]]
    screenshots: List[str]
    dom_summaries: List[Dict[str, Any]]
    axe_results: List[Dict[str, Any]]
    ux_findings: Dict[str, Any]
    accessibility_findings: Dict[str, Any]
    merged_summary: Dict[str, Any]
    narrative: Dict[str, Any]
    final_report_md: str
    execution_logs: List[str]
    browser_retries: int
    browser_error: Optional[str]
    human_decision: Optional[str]
    exported_files: Dict[str, str]
    start_time: float

def log_event(state: AppState, message: str) -> AppState:
    ts = datetime.utcnow().isoformat()
    logger.info(message)
    state.setdefault("execution_logs", []).append(f"[{ts}] {message}")
    return state

## Browser Agent (Playwright + axe-core)
Crawls up to `MAX_PAGES_TO_CRAWL` same-domain pages, screenshots each, runs axe-core for WCAG checks, and pulls DOM signals (headings, alt text, forms, link text) via BeautifulSoup.

In [5]:
AXE_CDN = "https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.9.1/axe.min.js"

@with_retry
async def crawl_site(url: str, max_pages: int = MAX_PAGES_TO_CRAWL) -> Dict[str, Any]:
    visited, screenshots, dom_summaries, axe_results = [], [], [], []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={"width": 1366, "height": 900})
        to_visit = [url]
        seen = set()
        base_domain = re.sub(r"^https?://", "", url).split("/")[0]

        while to_visit and len(visited) < max_pages:
            current = to_visit.pop(0)
            if current in seen:
                continue
            seen.add(current)
            page = await context.new_page()
            try:
                await page.goto(current, timeout=PAGE_TIMEOUT_MS, wait_until="networkidle")
                title = await page.title()
                html = await page.content()
                soup = BeautifulSoup(html, "html.parser")

                shot_path = SCREENSHOT_DIR / f"page_{len(visited)+1}.png"
                await page.screenshot(path=str(shot_path), full_page=True)
                screenshots.append(str(shot_path))

                try:
                    await page.add_script_tag(url=AXE_CDN)
                    axe_report = await page.evaluate("async () => await axe.run()")
                except Exception as axe_err:
                    axe_report = {"error": str(axe_err)}
                axe_results.append({"page": current, "report": axe_report})

                headings = [h.get_text(strip=True) for h in soup.find_all(re.compile("^h[1-6]$"))]
                links_missing_text = [a for a in soup.find_all("a") if not a.get_text(strip=True)]
                imgs_missing_alt = [img for img in soup.find_all("img") if not img.get("alt")]
                forms = soup.find_all("form")

                dom_summaries.append({
                    "page": current,
                    "title": title,
                    "num_headings": len(headings),
                    "headings": headings[:15],
                    "num_links": len(soup.find_all("a")),
                    "links_missing_text": len(links_missing_text),
                    "images_missing_alt": len(imgs_missing_alt),
                    "num_forms": len(forms),
                    "text_sample": mask_pii(soup.get_text(separator=" ", strip=True)[:2000]),
                })
                visited.append({"url": current, "title": title, "status": "ok"})

                for a in soup.find_all("a", href=True):
                    href = a["href"]
                    if href.startswith("http") and base_domain in href and href not in seen:
                        to_visit.append(href)
                    elif href.startswith("/"):
                        joined = url.rstrip("/") + href
                        if joined not in seen:
                            to_visit.append(joined)
            except Exception as e:
                visited.append({"url": current, "status": "error", "error": str(e)})
            finally:
                await page.close()

        await browser.close()
    return {
        "visited_pages": visited,
        "screenshots": screenshots,
        "dom_summaries": dom_summaries,
        "axe_results": axe_results,
    }

## LLM helper (structured JSON + injection sanitization)

In [6]:
def call_llm_json(system_prompt: str, user_prompt: str, retries: int = 1) -> Dict[str, Any]:
    """
    Call the LLM and parse a JSON response. Retries once on a malformed
    response before giving up -- this matters more now that we're routed
    through OpenRouter's free-model pool (`openrouter/free`), where each
    call can land on a different underlying model with different JSON
    reliability. On repeated failure, returns a dict flagged with
    'json_parse_failed': True instead of silently defaulting fields to 0
    downstream.
    """
    if detect_prompt_injection(user_prompt):
        logger.warning("Potential prompt injection detected in scraped content; sanitizing.")
        user_prompt = re.sub("|".join(INJECTION_PATTERNS), "[REDACTED]", user_prompt, flags=re.IGNORECASE)

    last_content = ""
    for attempt in range(retries + 1):
        response = llm.invoke([
            {"role": "system", "content": system_prompt + "\nRespond ONLY with valid JSON, no markdown fences."},
            {"role": "user", "content": user_prompt},
        ])
        content = response.content.strip()
        content = re.sub(r"^```json|```$", "", content, flags=re.MULTILINE).strip()
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            last_content = content
            if attempt < retries:
                logger.warning(f"LLM did not return valid JSON (attempt {attempt + 1}/{retries + 1}); retrying.")
            else:
                logger.error("LLM did not return valid JSON after retries; flagging as failed.")

    return {"raw_output": last_content, "json_parse_failed": True}

## Agents: Planner, UX Reviewer, Accessibility, Merge, Report Writer

In [7]:
def planner_agent(state: AppState) -> AppState:
    log_event(state, "Planner: creating crawl + evaluation plan")
    state["plan"] = call_llm_json(
        system_prompt="You are a senior QA planning agent for a UX/accessibility audit.",
        user_prompt=(f"Create a short crawl and evaluation plan (JSON keys 'crawl_strategy', "
                      f"'focus_areas', 'max_pages') for auditing this URL: {state['url']}. "
                      f"Keep max_pages <= {MAX_PAGES_TO_CRAWL}.")
    )
    return state

In [8]:
def run_async_in_thread(coro):
    """
    Run an async coroutine to completion on a fresh event loop in a
    dedicated thread. Keeps Playwright's asyncio usage fully isolated
    from whatever event loop Gradio/uvicorn is running on, so the two
    never collide (no nest_asyncio patching needed).
    """
    box = {}

    def runner():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["result"] = loop.run_until_complete(coro)
        except Exception as e:
            box["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=runner)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


def browser_node(state: AppState) -> AppState:
    log_event(state, f"Browser Agent: crawling {state['url']}")
    try:
        result = run_async_in_thread(crawl_site(state["url"]))
        state.update(result)
        state["browser_error"] = None
    except Exception as e:
        state["browser_error"] = str(e)
        state["browser_retries"] = state.get("browser_retries", 0) + 1
        log_event(state, f"Browser Agent failed: {e}")
    return state

def browser_router(state: AppState) -> str:
    if state.get("browser_error") and state.get("browser_retries", 0) <= MAX_RETRIES:
        return "retry"
    if state.get("browser_error"):
        return "fail"
    return "continue"

In [9]:
UX_CATEGORIES = ["Navigation", "Layout", "Visual Hierarchy", "Consistency", "Feedback",
                  "Forms", "Mobile Responsiveness", "Performance Perception", "Error Handling", "User Flow"]

def ux_reviewer_agent(state: AppState) -> AppState:
    log_event(state, "UX Reviewer Agent: scoring UX categories")
    summary_input = json.dumps(state.get("dom_summaries", []), indent=2)[:6000]
    state["ux_findings"] = call_llm_json(
        system_prompt=(f"You are a senior UX auditor. Score each of these categories 1-10 with a short "
                        f"justification: {UX_CATEGORIES}. Return JSON: "
                        f"{{'scores': {{category: {{'score': int, 'justification': str}}}}, 'summary': str}}."),
        user_prompt=f"DOM/page summaries collected during crawl:\n{summary_input}"
    )
    return state

In [10]:
def accessibility_agent(state: AppState) -> AppState:
    log_event(state, "Accessibility Agent: analyzing WCAG 2.2 compliance")
    axe_results = state.get("axe_results", [])
    violations_summary = []
    for entry in axe_results:
        report = entry.get("report", {})
        for v in (report.get("violations", []) if isinstance(report, dict) else []):
            violations_summary.append({
                "page": entry["page"], "id": v.get("id"), "impact": v.get("impact"),
                "description": v.get("description"), "help": v.get("help"),
                "nodes_affected": len(v.get("nodes", [])),
            })
    result = call_llm_json(
        system_prompt=("You are a WCAG 2.2 accessibility auditor. Given raw axe-core violations, group them "
                        "by severity (critical/serious/moderate/minor), summarize root causes, and give concrete "
                        "remediation steps. Also comment on heading hierarchy, alt text, ARIA usage, keyboard/"
                        "focus, and form labeling based on the dom summaries. Return JSON: "
                        "{'severity_matrix': {...}, 'findings': [...], 'summary': str}."),
        user_prompt=(f"axe-core violations:\n{json.dumps(violations_summary, indent=2)[:6000]}\n\n"
                      f"DOM summaries:\n{json.dumps(state.get('dom_summaries', []), indent=2)[:4000]}")
    )
    result["raw_violation_count"] = len(violations_summary)
    state["accessibility_findings"] = result
    return state

In [11]:
def merge_findings_node(state: AppState) -> AppState:
    log_event(state, "Merging UX and Accessibility findings")

    ux_result = state.get("ux_findings", {}) or {}
    ux_scores = ux_result.get("scores", {})
    ux_ok = bool(ux_scores) and not ux_result.get("json_parse_failed")
    avg_ux = round(sum(v.get("score", 0) for v in ux_scores.values()) / len(ux_scores), 2) if ux_ok else None

    acc_result = state.get("accessibility_findings", {}) or {}
    severity_matrix = acc_result.get("severity_matrix", {})
    acc_ok = bool(severity_matrix) and not acc_result.get("json_parse_failed")
    critical = severity_matrix.get("critical", []) if acc_ok else []
    critical_count = (len(critical) if isinstance(critical, list) else critical) if acc_ok else None

    # Flag (rather than silently zero-out) rounds where the LLM response was
    # malformed, so a dropped score reads as "data quality issue this round"
    # instead of "the app actually got worse."
    data_quality_notes = []
    if not ux_ok:
        data_quality_notes.append("UX scoring response was malformed this round -- average UX score is not reliable.")
    if not acc_ok:
        data_quality_notes.append("Accessibility response was malformed this round -- critical-issue count is not reliable.")

    state["merged_summary"] = {
        "average_ux_score": avg_ux if avg_ux is not None else "N/A",
        "accessibility_critical_issues": critical_count if critical_count is not None else "N/A",
        "pages_reviewed": len(state.get("visited_pages", [])),
        "data_quality_notes": data_quality_notes,
    }
    return state

In [12]:
def report_writer_agent(state: AppState) -> AppState:
    log_event(state, "Report Writer Agent: compiling final report")
    context = {
        "url": state["url"],
        "pages_reviewed": len(state.get("visited_pages", [])),
        "ux_findings": state.get("ux_findings", {}),
        "accessibility_findings": state.get("accessibility_findings", {}),
        "merged_summary": state.get("merged_summary", {}),
    }
    narrative = call_llm_json(
        system_prompt=("You are a senior product report writer. Write an executive summary (3-5 sentences) "
                        "and a final conclusion (2-3 sentences) for a UX/accessibility audit, based on the "
                        "findings JSON. Return JSON: {'executive_summary': str, 'final_conclusion': str, "
                        "'overall_score_out_of_100': int}."),
        user_prompt=json.dumps(context, indent=2)[:6000]
    )

    md_lines = []
    md_lines.append("# AI App Reviewer — UX & Accessibility Audit\n")
    md_lines.append(f"**URL audited:** {state['url']}  \n**Date:** {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}\n")
    md_lines.append("## Executive Summary\n")
    md_lines.append(narrative.get("executive_summary", "N/A") + "\n")
    md_lines.append(f"**Overall Score:** {narrative.get('overall_score_out_of_100', 'N/A')}/100\n")

    md_lines.append("## UX Findings\n")
    for cat, data in state.get("ux_findings", {}).get("scores", {}).items():
        md_lines.append(f"- **{cat}**: {data.get('score')}/10 — {data.get('justification')}")
    md_lines.append("")

    md_lines.append("## Accessibility Findings\n")
    acc = state.get("accessibility_findings", {})
    md_lines.append(acc.get("summary", "N/A") + "\n")
    md_lines.append("### Severity Matrix\n")
    md_lines.append("```json\n" + json.dumps(acc.get("severity_matrix", {}), indent=2) + "\n```\n")

    md_lines.append("## Screenshots\n")
    for shot in state.get("screenshots", []):
        md_lines.append(f"![screenshot]({shot})")
    md_lines.append("")

    md_lines.append("## Prioritized Recommendations\n")
    for i, f in enumerate(acc.get("findings", []), 1):
        md_lines.append(f"{i}. {f}")
    md_lines.append("")

    md_lines.append("## Final Conclusion\n")
    md_lines.append(narrative.get("final_conclusion", "N/A"))

    state["final_report_md"] = "\n".join(md_lines)
    state["narrative"] = narrative
    return state

## Human-in-the-loop gate

Two ways to supply the decision:
- **Interactive/CLI use** — if `human_decision` isn't already set on the state, this node falls back to
  `input()` and asks right in the cell output.
- **Gradio UI use** — the graph is compiled with `interrupt_before=["human_approval"]`, so execution
  pauses *before* this node runs. The UI sets `human_decision` via `update_state()` and then resumes the
  graph, so this node just logs the decision it finds and moves on — no blocking `input()` call happens.

In [13]:
def human_approval_node(state: AppState) -> AppState:
    if state.get("human_decision"):
        log_event(state, f"Human-in-the-loop: decision received from UI = {state['human_decision']}")
        return state

    log_event(state, "Human-in-the-loop: awaiting reviewer decision (CLI fallback)")
    print("\n" + "=" * 70)
    print("HUMAN REVIEW — Summary before export")
    print("=" * 70)
    print(state.get("merged_summary", {}))
    print("\nExecutive summary preview:")
    print(state.get("narrative", {}).get("executive_summary", ""))
    decision = input("\nApprove report? (approve/reject/recrawl): ").strip().lower()
    if decision not in ("approve", "reject", "recrawl"):
        decision = "approve"
    state["human_decision"] = decision
    return state

def human_router(state: AppState) -> str:
    decision = state.get("human_decision", "approve")
    if decision == "recrawl":
        return "recrawl"
    if decision == "reject":
        return "rejected"
    return "approved"

## Export Agent (Markdown, JSON, PDF, logs)

In [14]:
def export_node(state: AppState) -> AppState:
    log_event(state, "Export Agent: writing markdown, JSON, PDF")
    stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    md_path = REPORT_DIR / f"report_{stamp}.md"
    md_path.write_text(state.get("final_report_md", ""), encoding="utf-8")

    json_path = REPORT_DIR / f"report_{stamp}.json"
    json_safe_state = {k: v for k, v in state.items() if k != "axe_results"}
    json_path.write_text(json.dumps(json_safe_state, indent=2, default=str), encoding="utf-8")

    pdf_path = REPORT_DIR / f"report_{stamp}.pdf"
    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(str(pdf_path), pagesize=letter)
    story = [Paragraph(state["url"], styles["Title"]), Spacer(1, 12)]
    for line in state.get("final_report_md", "").split("\n"):
        if line.strip():
            safe_line = line.replace("<", "&lt;").replace(">", "&gt;")
            story.append(Paragraph(safe_line, styles["Normal"]))
            story.append(Spacer(1, 6))
    doc.build(story)

    log_path = LOG_DIR / f"log_{stamp}.txt"
    log_path.write_text("\n".join(state.get("execution_logs", [])), encoding="utf-8")

    state["exported_files"] = {
        "markdown": str(md_path), "json": str(json_path),
        "pdf": str(pdf_path), "log": str(log_path),
    }
    logger.info(f"Exported: {state['exported_files']}")
    return state

def rejected_node(state: AppState) -> AppState:
    log_event(state, "Report rejected by human reviewer. Workflow ended without export.")
    return state

## Build the LangGraph state graph

SQLite-backed checkpointer, conditional edges for browser retries and the human decision.
`interrupt_before=["human_approval"]` is what lets the Gradio UI pause the graph, show the report, and
resume it after the reviewer clicks a button — instead of the graph blocking on `input()`.

In [15]:
checkpointer_conn = sqlite3.connect("ai_app_reviewer_checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(checkpointer_conn)

graph = StateGraph(AppState)
graph.add_node("planner", planner_agent)
graph.add_node("browser", browser_node)
graph.add_node("ux_review", ux_reviewer_agent)
graph.add_node("accessibility_review", accessibility_agent)
graph.add_node("merge_findings", merge_findings_node)
graph.add_node("report_generation", report_writer_agent)
graph.add_node("human_approval", human_approval_node)
graph.add_node("export", export_node)
graph.add_node("rejected", rejected_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "browser")
graph.add_conditional_edges("browser", browser_router, {
    "retry": "browser", "continue": "ux_review", "fail": "rejected",
})
graph.add_edge("ux_review", "accessibility_review")
graph.add_edge("accessibility_review", "merge_findings")
graph.add_edge("merge_findings", "report_generation")
graph.add_edge("report_generation", "human_approval")
graph.add_conditional_edges("human_approval", human_router, {
    "approved": "export", "rejected": "rejected", "recrawl": "browser",
})
graph.add_edge("export", END)
graph.add_edge("rejected", END)

app_graph = graph.compile(checkpointer=checkpointer, interrupt_before=["human_approval"])

## Optional: run an audit without the UI (CLI-style, unchanged behavior)

Skip this cell if you're only using the Gradio app below — it's kept for scripted/non-interactive use.
Because the graph now interrupts before `human_approval`, this cell resumes the graph itself so it still
behaves like a single end-to-end run with an `input()` prompt in the middle.

In [16]:
RUN_CLI_DEMO = False  # set True to try the non-UI flow
TARGET_URL = "https://example.com"  # <-- replace with the app URL to audit

if RUN_CLI_DEMO:
    validated_url = validate_url(TARGET_URL)
    initial_state: AppState = {
        "url": validated_url,
        "visited_pages": [], "screenshots": [], "dom_summaries": [],
        "execution_logs": [], "browser_retries": 0, "start_time": time.time(),
    }
    config = {"configurable": {"thread_id": f"audit-{int(time.time())}"}}
    state = app_graph.invoke(initial_state, config=config)          # pauses before human_approval
    state = app_graph.invoke(None, config=config)                   # resumes -> asks via input() -> exports

    print("\n--- DONE ---")
    print("Exported files:", state.get("exported_files"))
    print("Total pages reviewed:", len(state.get("visited_pages", [])))
    print(f"Elapsed: {time.time() - state['start_time']:.1f}s")

## Gradio Web Interface

- **Run Audit** — invokes the graph from `planner` through `report_generation`, then the graph pauses
  (`interrupt_before=["human_approval"]`) before the human gate. The summary and report preview are shown.
- **Approve / Reject / Re-crawl** — sets `human_decision` on the paused graph's state via
  `update_state()` and resumes it with `invoke(None, ...)`. Approve exports Markdown/JSON/PDF; Reject ends
  the run with no export; Re-crawl sends it back through the `browser` node and pauses again for another
  review round.

In [17]:
def _quality_suffix(summary: Dict[str, Any]) -> str:
    """Build a warning suffix if this round's LLM output looked malformed."""
    notes = summary.get("data_quality_notes", [])
    if not notes:
        return ""
    return "\n\n⚠️ " + " ".join(notes) + " (This is usually free-model flakiness/rate-limiting on openrouter/free -- try Re-crawl again.)"


def start_audit(url: str):
    try:
        validated_url = validate_url(url)
    except ValueError as e:
        return None, f"❌ {e}", "", gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    thread_id = f"audit-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}
    initial_state: AppState = {
        "url": validated_url,
        "visited_pages": [], "screenshots": [], "dom_summaries": [],
        "execution_logs": [], "browser_retries": 0, "start_time": time.time(),
    }

    try:
        state = app_graph.invoke(initial_state, config=config)  # runs planner..report_generation, then pauses
    except Exception as e:
        return None, f"❌ Audit failed: {e}", "", gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    if state.get("browser_error"):
        return None, f"❌ Crawl failed after retries: {state['browser_error']}", "", gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    summary = state.get("merged_summary", {})
    narrative = state.get("narrative", {})
    status_msg = (
        f"Pages reviewed: {summary.get('pages_reviewed')}\n"
        f"Average UX score: {summary.get('average_ux_score')}/10\n"
        f"Critical accessibility issues: {summary.get('accessibility_critical_issues')}\n"
        f"Overall score: {narrative.get('overall_score_out_of_100', 'N/A')}/100\n\n"
        f"Executive summary:\n{narrative.get('executive_summary', '')}\n\n"
        "Review the report preview below, then Approve, Reject, or Re-crawl."
        f"{_quality_suffix(summary)}"
    )
    preview = state.get("final_report_md", "")
    return thread_id, status_msg, preview, gr.update(interactive=True), gr.update(interactive=True), gr.update(interactive=True)


def resume_with_decision(thread_id: str, decision: str):
    if not thread_id:
        return "⚠️ Run an audit first.", "", gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    config = {"configurable": {"thread_id": thread_id}}
    app_graph.update_state(config, {"human_decision": decision})
    state = app_graph.invoke(None, config=config)

    if decision == "approve":
        files = state.get("exported_files", {})
        msg = (
            "✅ Approved & exported.\n"
            f"Markdown: {files.get('markdown')}\n"
            f"PDF: {files.get('pdf')}\n"
            f"JSON: {files.get('json')}\n"
            f"Log: {files.get('log')}"
        )
        return msg, state.get("final_report_md", ""), gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    if decision == "reject":
        return "❌ Rejected by reviewer. No files were exported.", "", gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=False)

    # recrawl -> graph looped back through browser and paused again before human_approval
    summary = state.get("merged_summary", {})
    narrative = state.get("narrative", {})
    msg = (
        "🔁 Re-crawled.\n"
        f"Pages reviewed: {summary.get('pages_reviewed')}\n"
        f"Average UX score: {summary.get('average_ux_score')}/10\n"
        f"Critical accessibility issues: {summary.get('accessibility_critical_issues')}\n\n"
        f"Executive summary:\n{narrative.get('executive_summary', '')}\n\n"
        "Ready for review again — Approve, Reject, or Re-crawl."
        f"{_quality_suffix(summary)}"
    )
    return msg, state.get("final_report_md", ""), gr.update(interactive=True), gr.update(interactive=True), gr.update(interactive=True)

In [19]:
custom_theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
    block_radius="16px",
    block_shadow="0 1px 3px rgba(0,0,0,0.08)",
)

custom_css = """
#header_banner {
    background: linear-gradient(135deg, #0f766e 0%, #0891b2 55%, #0284c7 100%);
    border-radius: 18px;
    padding: 22px 26px;
    margin-bottom: 12px;
}
#header_banner h1 { color: #ffffff !important; margin: 0 0 4px 0; font-size: 26px; }
#header_banner p  { color: #e0f2fe !important; margin: 0; font-size: 14px; }
#status_box textarea { font-family: var(--font-mono); font-size: 13px; line-height: 1.5; }
#report_preview { max-height: 480px; overflow-y: auto; padding-right: 6px; }
#action_row .lg { font-weight: 600; }
"""

with gr.Blocks(title="Inspectra", theme=custom_theme, css=custom_css) as demo:
    gr.HTML(
        "<div id='header_banner'>"
        "<h1>🔍 Inspectra</h1>"
        "<p>Enter a public URL. The agent pipeline (Planner → Browser/Playwright+axe-core → "
        "UX Reviewer → Accessibility → Merge → Report Writer) crawls and scores it, "
        "then pauses for your approval before exporting the report.</p>"
        "</div>"
    )

    thread_state = gr.State(value=None)

    with gr.Row():
        url_in = gr.Textbox(label="Application URL", placeholder="https://example.com", scale=4)
        run_btn = gr.Button("▶ Run Audit", variant="primary", scale=1)

    with gr.Row():
        with gr.Column(scale=1):
            status_out = gr.Textbox(label="Status / Summary", lines=14, interactive=False, elem_id="status_box")
        with gr.Column(scale=1):
            report_preview_out = gr.Markdown(label="Report Preview", elem_id="report_preview")

    with gr.Row(elem_id="action_row"):
        approve_btn = gr.Button("✅ Approve", variant="primary", interactive=False)
        reject_btn = gr.Button("❌ Reject", variant="stop", interactive=False)
        recrawl_btn = gr.Button("🔁 Re-crawl", variant="secondary", interactive=False)

    run_btn.click(
        fn=start_audit,
        inputs=[url_in],
        outputs=[thread_state, status_out, report_preview_out, approve_btn, reject_btn, recrawl_btn],
    )
    approve_btn.click(
        fn=lambda tid: resume_with_decision(tid, "approve"),
        inputs=[thread_state],
        outputs=[status_out, report_preview_out, approve_btn, reject_btn, recrawl_btn],
    )
    reject_btn.click(
        fn=lambda tid: resume_with_decision(tid, "reject"),
        inputs=[thread_state],
        outputs=[status_out, report_preview_out, approve_btn, reject_btn, recrawl_btn],
    )
    recrawl_btn.click(
        fn=lambda tid: resume_with_decision(tid, "recrawl"),
        inputs=[thread_state],
        outputs=[status_out, report_preview_out, approve_btn, reject_btn, recrawl_btn],
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_2428/3033184585.py:28: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Inspectra", theme=custom_theme, css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8956752c623303fd1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notes / known limitations
- **Single notebook, not the full enterprise repo.** Docker, `docker-compose`, a FastAPI service layer,
  and a test suite are still intentionally out of scope for a `.ipynb`.
- **axe-core injection can fail** on sites with a strict CSP that blocks external `<script>` tags — that
  page's accessibility scan will just carry an `{'error': ...}` payload instead of violations.
- **`gpt-4o` stand-in** — there's no `gpt-5.2` model; point `MODEL_NAME` at whatever your API key can use.
- **Crawl is same-domain, breadth-first, capped at `MAX_PAGES_TO_CRAWL`** — deep multi-level nav trees or
  JS-only SPA routing may need `wait_until` / click-through logic added to `crawl_site`.
- **The Gradio app and the CLI cell share the same compiled `app_graph`** — don't run the CLI demo and
  the UI against the *same* `thread_id` at once; each `start_audit()` call generates a fresh one, so this
  only matters if you hand-edit thread IDs.
